In [10]:
import pandas as pd
import numpy as np
import psycopg2
import sqlalchemy as db
from sqlalchemy import create_engine
import yaml

In [11]:
with open('C:\\Users\\dylan\\OneDrive\\Documentos\\mensajeria\\config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config['mensajeria']
    config_etl = config['ETL_PRO']

# Construct the database URL
url_co = (f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:"
          f"{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
# Create the SQLAlchemy Engine
mensajeria = create_engine(url_co)
etl_conn = create_engine(url_etl)

In [12]:
def generar_dim_hora() -> pd.DataFrame:
    
    segundos = pd.date_range(start='00:00:00', end='23:59:59', freq='1s')
    
    df = pd.DataFrame({'tiempo': segundos})
    
    # Clave sustituta: HHMMSS como entero (0 a 235959)
    df['key_hora'] = df['tiempo'].dt.strftime('%H%M%S').astype(int)
    
    # Atributos
    df['hora']         = df['tiempo'].dt.hour
    df['minutos']      = df['tiempo'].dt.minute
    df['segundos']     = df['tiempo'].dt.second
    df['hora_formato'] = df['tiempo'].dt.strftime('%H:%M:%S')
    df['hora_completa'] = df['tiempo'].dt.time
    
    # Franjas horarias
    df['franja'] = pd.cut(
        df['hora'],
        bins=  [0,  6, 12, 14, 18, 21, 24],
        labels=['madrugada', 'mañana', 'mediodía', 'tarde', 'noche', 'noche alta'],
        right=False
    )
    
    df['es_hora_habil'] = (df['hora'] >= 8) & (df['hora'] < 18)
    
    return df[['key_hora', 'hora', 'minutos', 'segundos','hora_completa',
               'hora_formato', 'franja', 'es_hora_habil']]


dim_hora = generar_dim_hora()
print(dim_hora.shape)   # (86400, 7) → 86.400 segundos en un día
print(dim_hora.tail())
print(dim_hora['hora_completa'].iloc[0])  # 00:00:00

(86400, 8)
       key_hora  hora  minutos  segundos hora_completa hora_formato  \
86395    235955    23       59        55      23:59:55     23:59:55   
86396    235956    23       59        56      23:59:56     23:59:56   
86397    235957    23       59        57      23:59:57     23:59:57   
86398    235958    23       59        58      23:59:58     23:59:58   
86399    235959    23       59        59      23:59:59     23:59:59   

           franja  es_hora_habil  
86395  noche alta          False  
86396  noche alta          False  
86397  noche alta          False  
86398  noche alta          False  
86399  noche alta          False  
00:00:00


In [13]:
dim_hora.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 86400 entries, 0 to 86399
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   key_hora       86400 non-null  int64   
 1   hora           86400 non-null  int32   
 2   minutos        86400 non-null  int32   
 3   segundos       86400 non-null  int32   
 4   hora_completa  86400 non-null  object  
 5   hora_formato   86400 non-null  object  
 6   franja         86400 non-null  category
 7   es_hora_habil  86400 non-null  bool    
dtypes: bool(1), category(1), int32(3), int64(1), object(2)
memory usage: 3.1+ MB


In [14]:
dim_hora.to_sql('dim_hora', etl_conn, if_exists='replace', index=False)

400